# TradeFlowGNN: Model Evaluation

This notebook loads the trained checkpoints and compares the GCN performance against baselines.

In [ ]:
# ── Colab/Notebook Setup ───────────────────────────────────────────
import sys
from pathlib import Path
import os

def setup_colab():
    if 'google.colab' in str(get_ipython()):
        print("Running in Google Colab. Setting up environment...")
        
        # 1. Install dependencies
        !pip install -q torch-geometric pytorch-lightning pyarrow pandas numpy matplotlib seaborn tqdm pyyaml
        
        # 2. Clone repository if not already present
        repo_name = "TradeFlowGCN"
        if not Path(repo_name).exists() and not Path("src").exists():
            !git clone https://github.com/tolyaho/TradeFlowGCN.git
        
        # 3. Enter repository directory
        if Path(repo_name).exists() and not Path("src").exists():
            os.chdir(repo_name)
            print(f"Changed directory to {os.getcwd()}")
            
        # 4. Pull latest changes (ensure we have the robust CSV selection)
        if Path(".git").exists():
            print("Pulling latest code...")
            !git pull
        
        # 5. Set PYTHONPATH
        if os.getcwd() not in sys.path: sys.path.append(os.getcwd())
        src_path = os.path.join(os.getcwd(), "src")
        if src_path not in sys.path: sys.path.append(src_path)

setup_colab()

# ── Robust Path Setup ──────────────────────────────────────────────
curr_path = Path(".").resolve()
root_path = curr_path
while not (root_path / "src").exists() and root_path.parent != root_path:
    root_path = root_path.parent

if (root_path / "src").exists():
    if str(root_path / "src") not in sys.path:
        sys.path.append(str(root_path / "src"))
    print(f"Project Root: {root_path}")
else:
    print(f"Warning: Could not find 'src' directory relative to {curr_path}")

## 0. Training (Optional)

Run the cells below to prepare data and train the model in Colab.

In [ ]:
# Ensure data is downloaded and identify the 1.25GB Gravity file correctly
!python scripts/download_data.py

In [ ]:
# Run training script directly
!python scripts/train.py --config configs/default.yaml

## 1. Find Checkpoints

In [ ]:
log_dir = root_path / "lightning_logs"
ckpt_files = list(log_dir.glob("**/checkpoints/*.ckpt"))

if not ckpt_files:
    print(f"No checkpoints found. Please run the training cell above first.")
else:
    latest_ckpt = sorted(ckpt_files, key=lambda p: p.stat().st_mtime)[-1]
    print(f"Latest checkpoint: {latest_ckpt}")

## 2. Load Model

In [ ]:
from trade_flow_gcn.models.gcn import TradeFlowGCN
from trade_flow_gcn.training.lightning_module import TradeFlowModule
from trade_flow_gcn.utils.config import load_config

# Load config
config = load_config(str(root_path / "configs/default.yaml"))

# Initialize model architecture
model = TradeFlowGCN(
    node_input_dim=len(config['data']['node_features']),
    edge_input_dim=len(config['data']['edge_features']),
    hidden_dim=64,
    num_gnn_layers=3
)

if 'latest_ckpt' in locals():
    module = TradeFlowModule.load_from_checkpoint(latest_ckpt, model=model)
    module.eval()
    print("Model loaded successfully.")